In [1]:
#!apt update
#!apt install unzip
!unzip -o ./opinrank+review+dataset.zip -d ./opinrankdataset

Archive:  ./opinrank+review+dataset.zip
 extracting: ./opinrankdataset/OpinRankDataset.pdf  
 extracting: ./opinrankdataset/OpinRankDataset.zip  


In [2]:
!unzip -o ./opinrankdataset/OpinRankDataset.zip -d ./opinrankdataset/OpinRankDataset

Archive:  ./opinrankdataset/OpinRankDataset.zip
  inflating: ./opinrankdataset/OpinRankDataset/OpinRankDataset.pdf  
  inflating: ./opinrankdataset/OpinRankDataset/hotels/beijing/china_beijing_aloft_beijing_haidian  
  inflating: ./opinrankdataset/OpinRankDataset/hotels/beijing/china_beijing_ascott_beijing  
  inflating: ./opinrankdataset/OpinRankDataset/hotels/beijing/china_beijing_autumn_garden_courtyard_hotel  
  inflating: ./opinrankdataset/OpinRankDataset/hotels/beijing/china_beijing_bamboo_garden_hotel  
  inflating: ./opinrankdataset/OpinRankDataset/hotels/beijing/china_beijing_beijing_century_towers  
  inflating: ./opinrankdataset/OpinRankDataset/hotels/beijing/china_beijing_beijing_dong_fang_hotel  
  inflating: ./opinrankdataset/OpinRankDataset/hotels/beijing/china_beijing_beijing_far_east_international_youth_hostel  
  inflating: ./opinrankdataset/OpinRankDataset/hotels/beijing/china_beijing_beijing_friendship_hotel_grand_building  
  inflating: ./opinrankdataset/OpinRankDa

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
import pandas as pd
import os
import plotly.express as px
import plotly
import seaborn as sns
import numpy as np
import tqdm
import datetime

pd.set_option('display.float_format', lambda x: '%.2f' % x)
pd.set_option('display.max_colwidth', 5000)

import plotly.io as pio
pio.templates.default = 'simple_white'
from bertopic import BERTopic

## Loading data

In [5]:
def parse_file(file_name):
    tmp_data = []
    with open(file_name, 'rb') as f:
        for line in f:
            items = line.decode('ISO-8859-1').strip().split('\t', maxsplit=2)
            tmp_data.append(
                {
                    'date': items[0],
                    'title': items[1] if len(items) > 1 else '',
                    'body': items[2] if len(items) > 2 else ''
                }
            )
    tmp_df = pd.DataFrame(tmp_data)
    tmp_df['file_name'] = file_name
    return tmp_df

In [6]:
tmp_dfs = []

for city in ['london']:
    for hotel in tqdm.tqdm(os.listdir(f"./opinrankdataset/OpinRankDataset/hotels/{city}")):
        file_name = f"./opinrankdataset/OpinRankDataset/hotels/{city}/{hotel}"
        tmp_dfs.append(parse_file(file_name))

df = pd.concat(tmp_dfs)
df.sample(5)

100%|██████████| 875/875 [00:00<00:00, 1821.01it/s]


,date,title,body,file_name
36,Dec 8 2008,Be Aware: Irons and Parking,"I loved this hotel, we stayed there for our first anniversary, Early December, '08.Everything was perfect with two set backs.1. The hotel only has 10 irons for the whole hotel. So you have to call reception and they go find where it is. A four star hotel with only 10 irons... hmmm. They offered to send our clothes to be pressed complimentary but it was going to take half an hour to an hour. When the guy at the reception desk couldn't locate a free iron he apologised and asked if there was anything else he could do. I replied 'yeah, could someone go to Argos and buy an iron' to which he replied 'Sure, it'll be about 10 minutes'. Unfortunately me and my husband had been delayed long enough and couldn't wait for it to arrive... but it's good to know they we're willing to go the extra mile.2. ParkingIt cost £20 to park the car in the hotel overnight or £2 per hour according to the sign. We parked there for less than an hour and then went out on the night and on our return parked in the residential area which had very limited parking and is only available from 10pm till 9am. When we checked out of the hotel we didn't have to pay anything. So either they forgot to charge us or they start charging after an hour or they don't want to encourage parking in the hotel as they have very limited parking space available. Oh and once in the car parking grounds you have to go through the escape stairs to enter the hotel... which we found a bit weird.But on a whole it was a great experience. The breakfast was very filling and the staff were very attentive and asking if everything was ok or if we needed anything. Didn't dine or get to use the leisure facilities (but from other reviews those seem great) due to time constraints but will definitely be booking another stay in the hotel in the near future to truly experience everything it has to offer.",./opinrankdataset/OpinRankDataset/hotels/london/uk_england_london_crown_moran_hotel
18,Nov 21 2008,Refurbishment Transformation,"I stayed at the Crowne Plaza Ealing 18 times between March and August 2008 whilst my employer was relocating an office, and watched the hotel transform into a very solid 4* property.Rooms were absolutely spotless and very well designed. Standard rooms have shower/bath combination and old-style (surprisingly) TV, plus tea/coffee/mineral water. Executive rooms have a larger bathroom to provide a separate shower &amp; bath, but this reduces the size of the bedroom. Other exec differences include flat screen TV, orange juice in the fridge and access to club lounge (which had the smallest selection of snacks I've ever seen!). Worst point was internet cost £20 if you used for more than a few minutes.The basement restaurant and bar had excellent food (breakfast probably the best I've had in the UK) at fairly reasonable prices. The staff in both the restaurant/bar and reception were very friendly and helpful.In terms of location, the hotel was within walking distance of the office so ideal for me, but the general surroundings are not good. However, its proximity to the Central Line made an evening out in central London only 20 minutes away. Will definitely return on a leisure trip at some stage - a cost-effective way of having a weekend in London.",./opinrankdataset/OpinRankDataset/hotels/london/uk_england_london_crowne_plaza_hotel_london_ealing
122,Aug 16 2007,Ideal Location,Stayed here for a long weekend to see some shows.Location was perfect for Theatreland - the shows we saw - Lord of the Rings (MUST SEE) and Fiddler on the Roof were about 10min away.Just round corner from Trafalgar Square &amp; Buckingham Palace and Westminster within a short walk.Built over the Charing Cross Station but not too noisy at all.We upgraded to an Executive room on arrival for £35 per night - included a bottle of wine each day and was a very comfortable room which we thought was worth the extra - we effectively paid a total of abo

In [7]:
def get_hotel_chain(x):
    """Extract hotel chain from file name
    """
    if x.startswith('london_holiday_inn'):
        return 'Holiday Inn'
    if x.startswith('london_park_plaza'):
        return 'Park Plaza'
    if x.startswith('london_millennium'):
        return 'Millemiun'
    if x.startswith('london_park_inn'):
        return 'Park Inn'
    if x.startswith('london_hilton'):
        return 'Hilton'
    if x.startswith('london_radisson'):
        return 'Radisson'
    if x.startswith('london_travelodge'):
        return 'Travelodge'
    
    return 'other'

In [8]:
for p in ['title', 'body']:
    df[p] = df[p].str.strip()

# Extract country, city, hotel chain and hotel name from file name
df['country'] = df.file_name.apply(lambda x: x.split('/')[-1].split('_', 2)[0])
df['city'] = df.file_name.apply(lambda x: x.split('/')[-1].split('_', 2)[1])
df['full_hotel'] = df.file_name.apply(lambda x: x.split('/')[-1].split('_', 2)[2])
df['hotel'] = df.full_hotel.apply(lambda x: get_hotel_chain(x))
df.sample(3)

,date,title,body,file_name,country,city,full_hotel,hotel
43,May 15 2004,problems,"I was supposed to stay at the Mabledon between 05/05/04 and 09/05/04 yet only had the room for me for one night. I had similar problems as the previous tripadvisor reviewer. I'd booked my room 6months before and had re confirmed the booking 2 weks before arrival only to find they had no room booked for me, they didnt know who i was, a great welcome after an 11 hour flight at ten oclock at night? They could only accommodate us for one night, they booked us into another hotel off of Russell Square but when i asked if they'd pay for the taxi there the receptionist just said the manager would need to authorise that, and like the previous reviewer, he was never there!Had they have aknowledged responsibility i wouldnt be writing this as the staff were great and overall a decent hotel but it seems like somethings not working there! Be careful or better still dont risk it, lots of other hotels in the area!",./opinrankdataset/OpinRankDataset/hotels/london/uk_england_london_mabledon_court_hotel,uk,england,london_mabledon_court_hotel,other
37,Aug 21 2007,WOW!,"From the elegant entrance to our spectacular suite,The Halkin was first class all the way! The locationwas central to many of the tourist attractions that wewanted to see, and the staff was polished andprofessional. The modern design somehow meldedbeautifully with the historical surroundings. The roomeven had a computer system that controlled everythingfrom the lights to the ac, and even maid or butlerservice!We did not try the in house restaurant (Thai food),but the room service was to die for (mozzarella andpesto burger, anyone?).The only downside was the high price tag, but theluxury that the hotel provided was well worth it forthis once in a lifetime European vacation.",./opinrankdataset/OpinRankDataset/hotels/london/uk_england_london_the_halkin,uk,england,london_the_halkin,other
8,Jan 7 2009,Would Not Return,"Plus 1: I got down late for breakfast on my first morning. The manager was very nice and organized a late breakfast for me. He was very nice and friendly at that time.Plus 2: I complained about my small room, and for the same price, I was moved to a bigger room a couple of days later. They did not seem keen on the idea of me checking out early.Plus 3: The hotel was very warm. They did not skimp on the heating which was great.Plus 4: The breakfast was very nice. Always a nice hot cuppa tea. They used real china and the service was prompt and attentive. It was awkward sharing a table with total strangers but it was either that or go hungry.Minus 1: My room was very narrow and the bathroom was very cramped in a cupboard in which I had to step up in order to get onto the toilet. The basin had nasty chips and was unhygienic looking. Minus 2: The bed was comfy but the pillow was not! I had to use my clothes as a pillow. However, when I changed rooms, the next pillow was comfortable.Minus 3: The attitude of the male staff seemed to change after my complaint about the small room and bathroom. They seemed to become cold and unfriendly.I would not stay here again but the location near Victoria was great. The Comfort Inn is closer still to Victoria and if I were ever to stay in this part of London again, I would choose the Comfort Inn.",./opinrankdataset/OpinRankDataset/hotels/london/uk_england_london_jubilee_hotel,uk,england,london_jubilee_hotel,other


In [9]:
# How many reviews do we have per hotel?
df.groupby(['hotel'], as_index = False).aggregate({'full_hotel': ['count', 'nunique']})\
    .sort_values(('full_hotel', 'count'), ascending = False)

hotel full_hotel        
                    count nunique
7        other      66459     793
1  Holiday Inn       2844      26
0       Hilton       2671      11
6   Travelodge       2178      22
5     Radisson       2103      12
4   Park Plaza       1346       4
2    Millemiun       1209       5
3     Park Inn        539       2

In [10]:
# We have some reviews that do not belong to any of the 7 hotel chains. Let's remove them for now.
df = df[df.hotel != 'other']

In [11]:
# Which hotels have the most reviews?
df[df.country == 'uk'].hotel.value_counts().head(30)

hotel
Holiday Inn    2844
Hilton         2671
Travelodge     2178
Radisson       2103
Park Plaza     1346
Millemiun      1209
Park Inn        539
Name: count, dtype: int64

In [12]:
# How many duplicate reviews do we have?
df.shape[0], df.drop_duplicates().shape[0]

(12890, 12890)

In [13]:
df['id'] = list(range(df.shape[0]))
df['review'] = df.title + ' ' + df.body
df = df[['id', 'hotel', 'review']]

In [14]:
"""Let's visualize the number of reviews per hotel chain. We can see that the Holiday Inn and Park Plaza have the most reviews, while the Travelodge has the least.
"""

colormap = px.colors.qualitative.Safe

fig = px.bar(
    df.groupby('hotel')[['id']].count()\
        .sort_values('id', ascending = False),
    text_auto = 'd',
    title = 'Reviews for London Hotels',
    labels = {'value': 'number of reviews'}
)

fig.update_traces(marker_color=colormap[6],
                  marker_line_color=colormap[6],
                  marker_line_width=1.5,
                  opacity=0.9)

fig.update_traces(textfont_size=12, textangle=0, textposition="outside", cliponaxis=False) 

fig.update_layout(showlegend = False)

## Translate Comments

In [15]:
import langdetect
import json
from deep_translator import GoogleTranslator

In [20]:
def detect_language(text):
    try:
        return langdetect.detect(text)
    except KeyboardInterrupt as e:
        raise(e)
    except:
        return '<-- ERROR -->'

def get_translation(text):
    try:
        source_lang = detect_language(text)
        if source_lang != 'en':
            return GoogleTranslator(source=source_lang, target='en').translate(str(text))
        else:
            return str(text)
    except KeyboardInterrupt as e:
        raise(e)
    except:
        return '<-- ERROR -->'
    
tmp_data = []

for rec in tqdm.tqdm(df.to_dict('records')):
    tmp_data.append(
        {'id': rec['id'],
        'lang': detect_language(rec['review']),
        'reviews_transl': get_translation(rec['review'])}
    )

100%|██████████| 12890/12890 [06:56<00:00, 30.95it/s] 


In [21]:
df = df.merge(pd.DataFrame(tmp_data))

In [22]:
df.to_csv('raw_transl_data.csv', index = False, sep = '\t')

In [23]:
df[df.lang != 'en'].sample(5)

,id,hotel,review,lang,reviews_transl
9952,10145,Radisson,perfecta ubicacion,es,perfect location
478,484,Millemiun,Wo sind die Millionen der angeblichen Renovierung?,de,Where are the millions from the alleged renovation?
756,767,Travelodge,Ottima posizione pulizia buonaprezzo interessante,it,"Excellent location, cleanliness, good, interesting price"
171,171,Holiday Inn,buen lugar si visitas el Camden,es,good place if you visit Camden
1172,1195,Holiday Inn,correct confortable excentré,fr,correct comfortable off-center


## Clearing not meaningful comments

In [24]:
df['reviews_transl'] = df.reviews_transl.str.strip()
df['reviews_len'] = df.reviews_transl.str.len()

In [25]:
# レビューの文字数の分布を可視化. おおよそ 5 % のレビューが 20 文字以下あることがわかる。
fig = px.histogram(df,
                   x='reviews_len',
                   nbins=250,
                   range_x=[0, 500],
                   histnorm='percent',
                   labels={'reviews_len': 'number of characters', 'percent': 'share of reviews, %'},
                   title='Number of characters in reviews')

fig.update_traces(marker_color=colormap[6], opacity=0.9)
fig

In [26]:
# レビューの文字数の分布をホテルチェーン別に可視化.
fig = px.histogram(df,
                   x='reviews_len',
                   color = 'hotel',
                   nbins=500,
                   range_x=[0, 1000],
                   labels={'reviews_len': 'number of characters', 'percent': 'share of reviews, %'},
                   title='Number of characters in reviews by hotel chain')
fig

In [27]:
# レビューを頻度順にソートして、上位 20 件を表示.
df.reviews_transl.str.lower().str.strip().value_counts().head(20)

reviews_transl
great location                8
perfect                       8
excellent hotel               7
great hotel                   7
good value for money          6
very good hotel               6
excellent value for money     5
very nice hotel               5
good value for london         4
very nice                     4
great deal                    4
very good                     4
optimal                       4
excellent choice              3
perfect location              3
impeccable                    3
highly recommended            3
great hotel great location    3
great value                   3
good quality/price ratio      3
Name: count, dtype: int64

In [ ]:
# 20 文字以下のレビューは全体の約3%を占めていることがわかる。
print(df[df.reviews_len < 20].shape[0], df.shape[0])
print(df[df.reviews_len < 20].shape[0]/df.shape[0])

367 12651
0.029009564461307407


In [29]:
# 文字数が <= 20 のレビューと > 20 のレビューで、ホテルチェーンの分布を比較.
df['length_group'] = df.reviews_len.apply(lambda x: '<= 20' if x <= 20 else '> 20')

In [30]:
# レビュー長ごとにホテルチェーンの分布を集計して、ピボットテーブルを作成.
len_stats_df = df.pivot_table(
    index = 'hotel',
    values = 'id',
    columns = 'length_group',
    aggfunc = 'count'
)

# レビュー長ごとのレビュー数を合計して、total 列を作成. その後、total 列の値でソート.
len_stats_df['total'] = len_stats_df.sum(axis = 1)
len_stats_df = len_stats_df.sort_values('total', ascending = False)
len_stats_df

length_group,<= 20,> 20,total
hotel,,,
Holiday Inn,114,2659,2773
Hilton,59,2552,2611
Travelodge,75,2078,2153
Radisson,59,2018,2077
Park Plaza,39,1285,1324
Millemiun,35,1149,1184
Park Inn,21,508,529


In [31]:
# ホテルチェーンごとに、レビュー長グループの割合を可視化. 短いレビューの偏りは特に見られないことがわかる.
px.bar(
    len_stats_df.apply(lambda x: 100.*x/len_stats_df.total).drop('total', axis = 1),
    text_auto = '.2f',
    color_discrete_map = {
          '<= 20': colormap[2],
          '> 20': colormap[5]
      }, title = "Reviews' length by hotel",
    labels = {'value': 'share of reviews, %', 'course_id': 'course',
             'length_group': 'review length'}
)

In [32]:
df.sample(5)

,id,hotel,review,lang,reviews_transl,reviews_len,length_group
8259,8421,Hilton,"Excellent hotel staff very helpful and friendly food great. Stayed with my daughter for 3 nights in November (for shopping). Booked this hotel before reading the reviews. Was rather worried that I had made a big mistake after reading them! Couldnt have been more wrong. The staff at the hotel were exceedingly helpful and friendly. The rooms were were very clean and well stocked with complimentary coffee / tea and toiletries (which were all replenished daily). Cannot fault any of the service front of of house, restaurant or cleaning staff. The food was really good especially the breakfast. Would have no worries about recommending this hotel and would definately stay there again.",en,"Excellent hotel staff very helpful and friendly food great. Stayed with my daughter for 3 nights in November (for shopping). Booked this hotel before reading the reviews. Was rather worried that I had made a big mistake after reading them! Couldnt have been more wrong. The staff at the hotel were exceedingly helpful and friendly. The rooms were were very clean and well stocked with complimentary coffee / tea and toiletries (which were all replenished daily). Cannot fault any of the service front of of house, restaurant or cleaning staff. The food was really good especially the breakfast. Would have no worries about recommending this hotel and would definately stay there again.",685,> 20
7815,7971,Hilton,"First class in every respect I was with a group and we appeared to get an excellent deal. The hotel is excellent, modern and spotless with very friendly, helpful staff.Fabulous rooms with HUGE beds. Great choice at buffet breakfast.An excellentstay, would definitely go again",en,"First class in every respect I was with a group and we appeared to get an excellent deal. The hotel is excellent, modern and spotless with very friendly, helpful staff.Fabulous rooms with HUGE beds. Great choice at buffet breakfast.An excellentstay, would definitely go again",275,> 20
3271,3329,Holiday Inn,"Great Hotel Stayed at this hotel on Oct 5th 2005 for one night. Check-in was fast. Room was clean, bright. The location is also good because it is literally 150 footsteps (I counted for this review) from Gloucester Road Underground Station. Word of advise. If you are arriving via Gloucester Road Underground Station, turn right after exiting the station and right again up the street at the Baileys Hotel.Breakfast next morning was fine, but we had to wait a while for it to be served. There was a mix-up with my order. You can have any type of breakfast and as much as you like.I now have a hotel I will use again as a base in London for future visits.",en,"Great Hotel Stayed at this hotel on Oct 5th 2005 for one night. Check-in was fast. Room was clean, bright. The location is also good because it is literally 150 footsteps (I counted for this review) from Gloucester Road Underground Station. Word of advise. If you are arriving via Gloucester Road Underground Station, turn right after exiting the station and right again up the street at the Baileys Hotel.Breakfast next morning was fine, but we had to wait a while for it to be served. There was a mix-up with my order. You can have any type of breakfast and as much as you like.I now have a hotel I will use again as a base in London for future visits.",654,> 20
4233,4311,Radisson,"Great Location. Not-so-great Rooms. This place has a great location right where the action is in West London and Leicester Square. Don't confuse this hotel with the Radisson Edwardian Hampshire immediately opposite this hotel. This hotel has a nice lobby, friendly staff, however, that's where everything ends. The rooms are awful and dilapidated, the bathrooms look about 30 years old and smokey (yes you can smell smoke from other people's rooms even if you don't smoke), and the amenities just don't stack up for the price. In addition, there is NO AIR CONDITIONING, and these 

In [33]:
# 20 文字以下のレビューをフィルタリングして、length_group 列を削除.
filt_df = df[df.reviews_len > 20].drop('length_group', axis = 1)

In [34]:
filt_df.sample(5)

,id,hotel,review,lang,reviews_transl,reviews_len
9721,9913,Holiday Inn,Excellent stay for the price we paid!! Having wanted to travel to London for a mini break with two relatives we thought it would be difficult to find a room that would sleep 3 adults. After searching on superbreak we found a great deal for 2 nights stay at this hotel and dirty dancing theatre tickets. The hotel was around 25 minutes on the tube from the main sights of London but this was fine for us as we did not mind taking the tube and we had 2 full days to explore so 25 minutes was nothing. The hotel is in zone 3 of the tube so tickets are a bit more expensive but because of the great deal we got we did not mind paying that bit extra. The nearest tube station for the hotel is Colliers Wood which is on the Northern Line - the hotel is just across the road from the station. The hotel itself was great as it was very clean and this is the most important factor for us. The only negative comment i would make is that it was a little cramped as there was a double and a sofa bed pulled out on the floor due to the fact there was 3 of us. If there had of only been 2 of us then it would have been fine. This was not a major problem though as we spent so little time in the hotel room it was just somewhere to sleep really. Breakfast was lovely and there was a wide choice to choose from which we kept helping ourselves to!Overall i would go back to this hotel as it meeted our needs. If you want to be close to central London and within walkin distance of the sights then this would not be for you but if you do stay here i am sure you will be pleased with the hotel and its staff!!,en,Excellent stay for the price we paid!! Having wanted to travel to London for a mini break with two relatives we thought it would be difficult to find a room that would sleep 3 adults. After searching on superbreak we found a great deal for 2 nights stay at this hotel and dirty dancing theatre tickets. The hotel was around 25 minutes on the tube from the main sights of London but this was fine for us as we did not mind taking the tube and we had 2 full days to explore so 25 minutes was nothing. The hotel is in zone 3 of the tube so tickets are a bit more expensive but because of the great deal we got we did not mind paying that bit extra. The nearest tube station for the hotel is Colliers Wood which is on the Northern Line - the hotel is just across the road from the station. The hotel itself was great as it was very clean and this is the most important factor for us. The only negative comment i would make is that it was a little cramped as there was a double and a sofa bed pulled out on the floor due to the fact there was 3 of us. If there had of only been 2 of us then it would have been fine. This was not a major problem though as we spent so little time in the hotel room it was just somewhere to sleep really. Breakfast was lovely and there was a wide choice to choose from which we kept helping ourselves to!Overall i would go back to this hotel as it meeted our needs. If you want to be close to central London and within walkin distance of the sights then this would not be for you but if you do stay here i am sure you will be pleased with the hotel and its staff!!,1590
5705,5815,Radisson,"good location perfect for a sight seeing trip we arrived at the hotel early, around 12pm and they gave us a room immediately. a fab start after a long journey. the hotel was spacious actually - we had a king size bed and a 3 seater sofa and a lot of space really. I was quite shocked - in a good way. It is literally a 2 minute walk from the gloucester road underground station with quite a lot of local amenities such as a waitrose and tesco right nearby in case you want to buy some water and a sandwich, restaurants too. The breakfast at the hotel was also excellent - my boyfriend was very pleased that they had 'real cumberland sausages' not the shoddy cheap breakfast you can expect elsewhere - although the servi

In [35]:
# レビューの言語の分布を可視化.
fig = px.bar(
    100 * filt_df.lang.value_counts(normalize = True).head(10),
    text_auto = '.2f',
    labels = {'value': 'share of reviews, %', 'index': 'language'},
    title = 'Top reviews languages'
)

fig.update_layout(showlegend = False)

fig.update_traces(marker_color=colormap[6],
                  marker_line_color=colormap[6],
                  marker_line_width=1.5,
                  opacity=0.9)

fig.update_traces(textfont_size=12, textangle=0, textposition="outside", cliponaxis=False) 

In [36]:
# 20 文字以下のレビューを除外したデータセットを保存.
filt_df.to_csv('hotel_reviews_with_transl.csv', index = False, sep = '\t')